This creates the data used in the tutorials.

In [1]:
import numpy as np
import plopp as pp
import scipp as sc

import easydynamics.sample_model as sm
from easydynamics.sample_model import ComponentCollection
from easydynamics.sample_model import Gaussian
from easydynamics.sample_model.sample_model import SampleModel

%matplotlib widget

In [2]:
# Create data for tutorial0_basics.ipynb. Data is called fake_simple_data.hdf5.

Q = sc.linspace(start=0.2, stop=2.1, num=32, unit='1/angstrom', dim='Q')

component_collection = ComponentCollection()
component_collection.append_component(Gaussian(area=0.45, width=0.1))

model = SampleModel(components=component_collection, Q=Q)

for i in range(Q.size):
    components = model.get_component_collection(i)
    offset = 0.0
    components[0].area = 3.79 - 0.2 * Q[i].value

energy = sc.linspace(start=-3.0, stop=3.0, num=756, unit='meV', dim='energy')

intensity = sc.array(values=model.evaluate(x=energy), dims=['Q', 'energy'])

intensity_dataarray = sc.DataArray(data=intensity, coords={'Q': Q, 'energy': energy})

rng = np.random.default_rng()
noise = rng.normal(loc=0.0, scale=1.55, size=intensity_dataarray.shape)
intensity_dataarray.values += noise
intensity_dataarray.variances = noise**2

# sc.io.save_hdf5(intensity_dataarray, 'fake_simple_data.hdf5')
pp.slicer(intensity_dataarray)

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [3]:
# Create data for tutorial0_more_advanced.ipynb. Data is called fake_advanced_data.hdf5.
Q = sc.linspace(start=0.2, stop=2.1, num=32, unit='1/angstrom', dim='Q')

component_collection = sm.ComponentCollection()
component_collection.append_component(sm.Gaussian(area=0.45, width=0.05))
component_collection.append_component(sm.Lorentzian(area=10.45, width=0.4))
component_collection.append_component(
    sm.DampedHarmonicOscillator(area=1.45, width=0.1, center=1.4)
)
component_collection.append_component(sm.Polynomial(coefficients=[1.2, 0.2]))
model = sm.SampleModel(components=component_collection, Q=Q)


energy = sc.linspace(start=-3.0, stop=3.0, num=756, unit='meV', dim='energy')
intensity_values = np.zeros((Q.size, energy.size))
rng = np.random.default_rng()
noise = rng.normal(loc=0.0, scale=0.35, size=intensity_values.shape)

for i in range(Q.size):
    components = model.get_component_collection(i)
    offset = sc.scalar(value=rng.uniform(0.05, 0.15), unit='meV')
    components[0].area = 3.79 - 0.2 * Q[i].value
    components[2].center = 1.4 + Q[i].value / 10.0
    components[2].area = 2.45 + Q[i].value / 10.0

    intensity_values[i, :] = components.evaluate(x=energy.values - offset.value)


intensity = sc.array(values=intensity_values, dims=['Q', 'energy'])
intensity_dataarray = sc.DataArray(data=intensity, coords={'Q': Q, 'energy': energy})

noise = rng.normal(loc=0.0, scale=0.35, size=intensity_dataarray.shape)
intensity_dataarray.values += noise
intensity_dataarray.variances = noise**2

# sc.io.save_hdf5(intensity_dataarray, 'fake_advanced_data.hdf5')
pp.slicer(intensity_dataarray)

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [4]:
# Create data for tutorial3_multiple_data.ipynb. Data are called fake_fine_resolution_data.hdf5
# and fake_coarse_resolution_data.hdf5.

# The same sample is measured twice. It has three Lorentzians: L1 is narrow, the width of L2
# grows as Q**2 like Brownian diffusion, and L3 is broad. The two data sets differ in resolution,
# energy window, Q values, background and intensity scale.

# The measured line shapes are written down with the resolution included. A Lorentzian convolved
# with a Gaussian is a Voigt, and a Lorentzian convolved with a Lorentzian is a Lorentzian whose
# width is the sum of the two widths.


def create_multiple_data(
    Q: sc.Variable,
    energy: sc.Variable,
    gaussian_width: float,
    tail_fraction: float,
    tail_width: float,
    scale: float,
    background: float,
) -> sc.DataArray:
    intensity_values = np.zeros((Q.size, energy.size))
    for i in range(Q.size):
        sample = {'L1': (0.3, 0.01), 'L2': (1.0, 0.08 * Q[i].value ** 2), 'L3': (2.0, 4.0)}
        components = sm.ComponentCollection()
        for name, (area, width) in sample.items():
            # The part of the line seen through the Gaussian of the resolution
            components.append_component(
                sm.Voigt(
                    name=f'{name} Gaussian part',
                    area=(1 - tail_fraction) * area,
                    lorentzian_width=width,
                    gaussian_width=gaussian_width,
                )
            )
            # and the part seen through its Lorentzian tails
            if tail_fraction > 0:
                components.append_component(
                    sm.Lorentzian(
                        name=f'{name} tail part',
                        area=tail_fraction * area,
                        width=width + tail_width,
                    )
                )
        # The background is added after the scale. An Analysis scales its background too, so it
        # finds a background of background / scale.
        intensity_values[i, :] = scale * components.evaluate(x=energy.values) + background

    sigma = 0.01 * intensity_values.max()
    noise = rng.normal(loc=0.0, scale=sigma, size=intensity_values.shape)
    intensity = sc.array(
        values=intensity_values + noise,
        variances=np.full(intensity_values.shape, sigma**2),
        dims=['Q', 'energy'],
    )
    return sc.DataArray(data=intensity, coords={'Q': Q, 'energy': energy})


rng = np.random.default_rng()

# Fine resolution: a Gaussian of width 0.01 meV, in a window of +-1 meV
fine_dataarray = create_multiple_data(
    Q=sc.linspace(start=0.8, stop=2.2, num=8, unit='1/angstrom', dim='Q'),
    energy=sc.linspace(start=-1.0, stop=1.0, num=201, unit='meV', dim='energy'),
    gaussian_width=0.01,
    tail_fraction=0.0,
    tail_width=0.0,
    scale=1.0,
    background=0.02,
)

# Coarse resolution: a Gaussian of width 0.1 meV with 10% Lorentzian tails, in a window of
# +-16 meV, and 3.2 times more intense
coarse_dataarray = create_multiple_data(
    Q=sc.linspace(start=1.0, stop=2.4, num=8, unit='1/angstrom', dim='Q'),
    energy=sc.linspace(start=-16.0, stop=16.0, num=321, unit='meV', dim='energy'),
    gaussian_width=0.1,
    tail_fraction=0.1,
    tail_width=0.1,
    scale=3.2,
    background=0.1,
)

# sc.io.save_hdf5(fine_dataarray, 'fake_fine_resolution_data.hdf5')
# sc.io.save_hdf5(coarse_dataarray, 'fake_coarse_resolution_data.hdf5')
pp.slicer(coarse_dataarray)

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…